<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
import pandas as pd
df = pd.read_csv('https://github.com/ritakimani9-lang/machinelearning/raw/main/data/raw/content_refresh_anonymized.csv')

for col in ['impressions_90d', 'search_volume', 'word_count', 'ctr', 'avg_position']:
    s = df[col].dropna()
    print(f"{col}: mean={s.mean():.1f}  median={s.median():.1f}  max={s.max():.1f}  skew={s.skew():.2f}")

impressions_90d: mean=5200.4  median=731.0  max=517715.0  skew=11.38
search_volume: mean=158.9  median=10.0  max=74000.0  skew=26.02
word_count: mean=3107.8  median=2877.0  max=9546.0  skew=0.94
ctr: mean=0.5  median=0.1  max=100.0  skew=17.44
avg_position: mean=16.3  median=10.8  max=245.0  skew=1.98


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
t1 = df.groupby('word_count_tier')['is_declining_label'].agg(['mean', 'count'])
t1['mean'] = (t1['mean'] * 100).round(1)
print(t1.reindex(['<1000', '1000-2000', '2000-3500', '3500+']))

                 mean  count
word_count_tier             
<1000            20.7    973
1000-2000        55.6   3780
2000-3500        58.8  11263
3500+            59.7   6285


Hypothesis: short/thin content is more likely to decline. Verdict: OPPOSITE. Short content (<1000 words) actually declines far less (21%) than everything longer (56–60%). All buckets are well-powered, so this is trustworthy, not noise. Worth a caveat: the <1000 bucket is small as a share of the dataset (973 of 30,000) — it may represent a different content style (e.g. quick feedly-style pieces) rather than proof that "shorter is safer" in general.

In [4]:
t2 = df.groupby('age_tier')['is_declining_label'].agg(['mean', 'count'])
t2['mean'] = (t2['mean'] * 100).round(1)
print(t2)

          mean  count
age_tier             
181-365   51.5  11368
31-90     66.9    492
365+      42.6   6360
91-180    62.6  11780


Hypothesis: younger content is more volatile than older, established content. Verdict: CONFIRMED. A clean, monotonic decrease in decline rate as content ages (63%→52%→43%), on three large, trustworthy buckets. Makes intuitive sense — new content hasn't found its footing; content that's survived a long time has stabilized.

In [5]:
df['eng_bucket'] = pd.cut(df['engagement_rate'], bins=[-1,0,20,40,60,101], labels=['0','1-20','21-40','41-60','61-100'])
t3 = df.groupby('eng_bucket', observed=True)['is_declining_label'].agg(['mean', 'count'])
t3['mean'] = (t3['mean'] * 100).round(1)
print(t3)

            mean  count
eng_bucket             
0           54.4  21629
1-20        53.6   7647
21-40       58.0    445
41-60       51.5    171
61-100      44.4    108


Hypothesis: more engaged sessions mean less risk of decline. Verdict: FALSE. On the two buckets you can actually trust (21,629 and 7,647 rows), decline rate is flat — 54.4% vs 53.6%, essentially no difference. The three buckets that look more interesting are all tiny (445, 171, 108 rows) and shouldn't move your conclusion. This signal doesn't do useful work — a legitimate, useful "don't build on this" finding.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tier_median = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_underperform'] = (df['ctr'] < tier_median).astype(int)

# pooled
t4 = df.groupby('ctr_underperform')['is_declining_label'].agg(['mean', 'count'])
t4['mean'] = (t4['mean'] * 100).round(1)
print(t4)

# broken out by tier — this is the important check
t5 = df.groupby(['position_tier', 'ctr_underperform'], observed=True)['is_declining_label'].agg(['mean', 'count'])
t5['mean'] = (t5['mean'] * 100).round(1)
print(t5)

                  mean  count
ctr_underperform             
0                 50.3  16921
1                 59.3  13079
                                mean  count
position_tier ctr_underperform             
deep          0                 34.4   1319
page_1        0                 53.6   5944
              1                 60.3   5870
page_3_5      0                 59.8   3680
              1                 52.4   3562
striking      0                 57.5   3657
              1                 64.4   3647
top_3         0                 24.1   2321


erdict: MIXED. The rule's assumption holds cleanly for page_1 and striking (underperformers decline more, as expected, on well-powered groups). It reverses for page_3_5 — underperformers there actually decline less. And deep/top_3 couldn't even be tested: every row landed in one bucket because CTR is so skewed and tie-heavy (lots of exact-zero or exact-median values) in those tiers that "strictly below median" caught nobody. This is the pooled result quietly averaging over a real disagreement between subgroups — the pooled 59.3%-vs-50.3% number would have been misleading on its own.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should not treat the CTR-underperformance flag as universal. It's a reasonable signal for page-1 and "striking-distance" content — pages already ranking decently that are still underconverting are worth a look. It breaks down for deep or top-3 rankings, where CTR values are too skewed/tied to compare meaningfully, and it actually points the wrong way for mid-page-3-5 content. Practically: apply this flag only within the tiers where it's been shown to work, and use a different method (e.g. percentile among nonzero-CTR pages only) for the tiers where ties make "below median" meaningless.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.